We import the necessary modules, setup configuration, initialize paths and load the processed data

In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
from transformers import DistilBertTokenizer, get_linear_schedule_with_warmup

# Import your modularized architectures
import sys
sys.path.append('..') # Ensure the 'models' folder is discoverable
from models.lstm import BiLSTMClassifier
from models.transformers import UniversalTransformerClassifier

# Configuration
SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PROCESSED_DIR = '../data/processed'
MODELS_DIR = '../models/saved_weights'
os.makedirs(MODELS_DIR, exist_ok=True)

torch.manual_seed(SEED)
print(f"✅ Using device: {DEVICE}")

# Load Data
train_df = pd.read_csv(os.path.join(PROCESSED_DIR, 'train.csv'))
val_df = pd.read_csv(os.path.join(PROCESSED_DIR, 'val.csv'))

LABEL_COLS = [c for c in train_df.columns if c not in ['Argument ID', 'text_raw', 'text_clean']]

# We only need X and y for training
y_train = train_df[LABEL_COLS].values
y_val = val_df[LABEL_COLS].values

✅ Using device: cuda


We iniitialie the dataloaders for the BiLSTM model (text_clean) and for the transformer models (text_raw).

In [3]:
from collections import Counter

from src.data_utils import LSTMDataset, BERTDataset

# Initialize Loaders
train_loader_lstm = DataLoader(LSTMDataset(train_df['text_clean'].values, y_train, vocab), batch_size=64, shuffle=True)
val_loader_lstm   = DataLoader(LSTMDataset(val_df['text_clean'].values, y_val, vocab), batch_size=64)

train_loader_bert = DataLoader(BertDataset(train_df['text_raw'].values, y_train, tokenizer), batch_size=16, shuffle=True)
val_loader_bert   = DataLoader(BertDataset(val_df['text_raw'].values, y_val, tokenizer), batch_size=16)

print("✅ DataLoaders ready.")

✅ DataLoaders ready.


We define a training function to handle early stopping, validation, and the critical multi-label F1 calculation.

In [4]:
def train_engine(model, train_loader, val_loader, pos_weights, is_bert=False, epochs=10, lr=1e-3):
    model = model.to(DEVICE)
    # The crucial weighted loss to prevent 0.00 F1
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)
    
    # AdamW is better for both LSTMs and Transformers
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    
    if is_bert:
        total_steps = len(train_loader) * epochs
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps)

    best_f1 = 0
    best_state = None
    patience_counter = 0
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        
        for batch in train_loader:
            optimizer.zero_grad()
            
            if is_bert:
                input_ids = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                labels = batch['labels'].to(DEVICE)
                outputs = model(input_ids, mask)
            else:
                inputs, labels = batch
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                outputs = model(inputs)
                
            loss = criterion(outputs, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            if is_bert:
                scheduler.step()
                
            train_loss += loss.item()
            
        # Validation
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                if is_bert:
                    input_ids = batch['input_ids'].to(DEVICE)
                    mask = batch['attention_mask'].to(DEVICE)
                    labels = batch['labels'].to(DEVICE)
                    outputs = model(input_ids, mask)
                else:
                    inputs, labels = batch
                    inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                    outputs = model(inputs)
                
                # Threshold logic
                preds = (torch.sigmoid(outputs) > 0.5).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(labels.cpu().numpy())
                
        val_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
        print(f"Epoch {epoch+1}/{epochs} | Loss: {train_loss/len(train_loader):.4f} | Val F1: {val_f1:.4f}")
        
        # Early Stopping
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= 3:
                print("Early stopping triggered.")
                break
                
    model.load_state_dict(best_state)
    return model, best_f1

We fit the BiLSTM model. But before that, we make a crucial adjustment based on an issue we faced before. We manually create a penalty ratio, punishing the model harshly if it fails to classify rare labels. We also use the square root to avoid exploding gradients. With this adjustment, the model will prioritize minority labels and achieve a more balanced performance and better generalization.

In [9]:
# 1. Calculate Imbalance Weights
pos_counts = torch.tensor(y_train.sum(axis=0), dtype=torch.float)
neg_counts = len(y_train) - pos_counts
pos_weights = torch.sqrt(neg_counts / pos_counts).to(DEVICE)

# 2. Initialize and Train
print("--- Training BiLSTM ---")
lstm_model = BiLSTMClassifier(vocab_size=vocab.vocab_size)
trained_lstm, lstm_best_f1 = train_engine(
    lstm_model, train_loader_lstm, val_loader_lstm, 
    pos_weights=pos_weights, is_bert=False, epochs=20, lr=1e-3
)

# 3. Save Weights
torch.save(trained_lstm.state_dict(), os.path.join(MODELS_DIR, 'bilstm_best.pt'))
print(f"✅ BiLSTM Saved. Best Val F1: {lstm_best_f1:.4f}")

--- Training BiLSTM ---
Epoch 1/20 | Loss: 0.6547 | Val F1: 0.3099
Epoch 2/20 | Loss: 0.5607 | Val F1: 0.4079
Epoch 3/20 | Loss: 0.5100 | Val F1: 0.4363
Epoch 4/20 | Loss: 0.4711 | Val F1: 0.4500
Epoch 5/20 | Loss: 0.4313 | Val F1: 0.4711
Epoch 6/20 | Loss: 0.3885 | Val F1: 0.4677
Epoch 7/20 | Loss: 0.3462 | Val F1: 0.4778
Epoch 8/20 | Loss: 0.3060 | Val F1: 0.4538
Epoch 9/20 | Loss: 0.2675 | Val F1: 0.4529
Epoch 10/20 | Loss: 0.2331 | Val F1: 0.4608
Early stopping triggered.
✅ BiLSTM Saved. Best Val F1: 0.4778


We now fit the first transformer model: DistillBERT. We set it as the transformer baseline and run it before standard BERT as it is a lighter version, optimized through knowledge distillation and it should serve as the first reference for transformer performance.

In [10]:
import gc
from transformers import AutoTokenizer

print("--- Training DistilBERT ---")
# 1. Clean memory from the LSTM run
del lstm_model
gc.collect()
torch.cuda.empty_cache()

# 2. Setup model-specific tokenizer and loaders
model_name_distil = 'distilbert-base-uncased'
tokenizer_distil = AutoTokenizer.from_pretrained(model_name_distil)

train_loader_distil = DataLoader(BertDataset(train_df['text_raw'].values, y_train, tokenizer_distil), batch_size=16, shuffle=True)
val_loader_distil   = DataLoader(BertDataset(val_df['text_raw'].values, y_val, tokenizer_distil), batch_size=16)

# 3. Instantiate and Train
distil_model = UniversalTransformerClassifier(model_name=model_name_distil).to(DEVICE)
trained_distil, distil_best_f1 = train_engine(
    distil_model, train_loader_distil, val_loader_distil, 
    pos_weights=pos_weights, is_bert=True, epochs=10, lr=2e-5
)

# 4. Save
torch.save(trained_distil.state_dict(), os.path.join(MODELS_DIR, 'distilbert_best.pt'))
print(f"✅ DistilBERT Saved. Best Val F1: {distil_best_f1:.4f}")

--- Training DistilBERT ---


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/10 | Loss: 0.6696 | Val F1: 0.3976
Epoch 2/10 | Loss: 0.5323 | Val F1: 0.4767
Epoch 3/10 | Loss: 0.4727 | Val F1: 0.5205
Epoch 4/10 | Loss: 0.4280 | Val F1: 0.5280
Epoch 5/10 | Loss: 0.3903 | Val F1: 0.5261
Epoch 6/10 | Loss: 0.3585 | Val F1: 0.5275
Epoch 7/10 | Loss: 0.3323 | Val F1: 0.5236
Early stopping triggered.
✅ DistilBERT Saved. Best Val F1: 0.5280


Now we fit BERT, which gives us the baseline for contextual embeddings. It uses the same WordPiece tokenization as DistilBERT but has more layers and parameters.

In [11]:
print("--- Training BERT Base ---")
# 1. Clean memory
del distil_model, trained_distil
gc.collect()
torch.cuda.empty_cache()

# 2. Setup model-specific tokenizer and loaders
model_name_bert = 'bert-base-uncased'
tokenizer_bert = AutoTokenizer.from_pretrained(model_name_bert)

train_loader_bert = DataLoader(BertDataset(train_df['text_raw'].values, y_train, tokenizer_bert), batch_size=16, shuffle=True)
val_loader_bert   = DataLoader(BertDataset(val_df['text_raw'].values, y_val, tokenizer_bert), batch_size=16)

# 3. Instantiate and Train
bert_model = UniversalTransformerClassifier(model_name=model_name_bert).to(DEVICE)
trained_bert, bert_best_f1 = train_engine(
    bert_model, train_loader_bert, val_loader_bert, 
    pos_weights=pos_weights, is_bert=True, epochs=10, lr=2e-5
)

# 4. Save
torch.save(trained_bert.state_dict(), os.path.join(MODELS_DIR, 'bert_best.pt'))
print(f"✅ BERT Saved. Best Val F1: {bert_best_f1:.4f}")

--- Training BERT Base ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/10 | Loss: 0.6724 | Val F1: 0.3875
Epoch 2/10 | Loss: 0.5252 | Val F1: 0.4840
Epoch 3/10 | Loss: 0.4590 | Val F1: 0.5236
Epoch 4/10 | Loss: 0.4090 | Val F1: 0.5361
Epoch 5/10 | Loss: 0.3664 | Val F1: 0.5282
Epoch 6/10 | Loss: 0.3297 | Val F1: 0.5369
Epoch 7/10 | Loss: 0.2990 | Val F1: 0.5341
Epoch 8/10 | Loss: 0.2746 | Val F1: 0.5245
Epoch 9/10 | Loss: 0.2579 | Val F1: 0.5214
Early stopping triggered.
✅ BERT Saved. Best Val F1: 0.5369


Now we will fit RoBERTa (Robustly Optimized BERT Approach) which addresses the issue of BERT being undertrained. It uses Byte-Pair Encoding (BPE) instead of WordPiece, so it uses its own AutoTokenizer instance to build the correct DataLoader.

In [12]:
print("--- Training RoBERTa ---")
# 1. Clean memory
del bert_model, trained_bert
gc.collect()
torch.cuda.empty_cache()

# 2. Setup model-specific tokenizer and loaders
model_name_roberta = 'roberta-base'
tokenizer_roberta = AutoTokenizer.from_pretrained(model_name_roberta)

train_loader_roberta = DataLoader(BertDataset(train_df['text_raw'].values, y_train, tokenizer_roberta), batch_size=16, shuffle=True)
val_loader_roberta   = DataLoader(BertDataset(val_df['text_raw'].values, y_val, tokenizer_roberta), batch_size=16)

# 3. Instantiate and Train
roberta_model = UniversalTransformerClassifier(model_name=model_name_roberta).to(DEVICE)
trained_roberta, roberta_best_f1 = train_engine(
    roberta_model, train_loader_roberta, val_loader_roberta, 
    pos_weights=pos_weights, is_bert=True, epochs=10, lr=2e-5
)

# 4. Save
torch.save(trained_roberta.state_dict(), os.path.join(MODELS_DIR, 'roberta_best.pt'))
print(f"✅ RoBERTa Saved. Best Val F1: {roberta_best_f1:.4f}")

# 5. Clean memory
del roberta_model, trained_roberta
gc.collect()
torch.cuda.empty_cache()

--- Training RoBERTa ---


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/10 | Loss: 0.6764 | Val F1: 0.4430
Epoch 2/10 | Loss: 0.5198 | Val F1: 0.4951
Epoch 3/10 | Loss: 0.4557 | Val F1: 0.5369
Epoch 4/10 | Loss: 0.4078 | Val F1: 0.5292
Epoch 5/10 | Loss: 0.3684 | Val F1: 0.5505
Epoch 6/10 | Loss: 0.3349 | Val F1: 0.5479
Epoch 7/10 | Loss: 0.3042 | Val F1: 0.5386
Epoch 8/10 | Loss: 0.2813 | Val F1: 0.5427
Early stopping triggered.
✅ RoBERTa Saved. Best Val F1: 0.5505
